### Persiapan: SparkSession & Dataset

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession siap. Versi Spark:", spark.version)

26/10/10 07:42:21 WARN Utils: Your hostname, vmi3356457 resolves to a loopback address: 127.0.1.1; using 13.140.149.59 instead (on interface eth0)
26/10/10 07:42:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 07:42:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/10 07:42:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [5]:
np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


---
## A. EXTRACT
Membaca ketiga sumber data langsung dari HDFS.

In [6]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


---
## B. TRANSFORM — Join & Kolom Turunan

In [7]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

print("Jumlah baris setelah join:", df_gabungan.count())
df_gabungan.show(5)

Jumlah baris setelah join: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makana

---
## C. TRANSFORM — Window Function: Top-2 Menu Terlaris per Cabang

In [8]:
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang \
    .withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

+-------------------------+---------+---------------+---------+
|nama_cabang              |nama_menu|total_penjualan|peringkat|
+-------------------------+---------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |8160000        |2        |
+-------------------------+---------+---------------+---------+



---
## D. TRANSFORM — Spark SQL: Total Penjualan per Kepala Barista

In [9]:
df_gabungan.createOrReplaceTempView("gabungan")

penjualan_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



---
## E. LOAD — Simpan ke HDFS (Parquet, dipartisi `kategori_menu`)

In [10]:
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

# Verifikasi: baca ulang & hitung jumlah baris
df_verifikasi = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris terbaca dari Parquet:", df_verifikasi.count())
print("Jumlah baris data gabungan       :", df_gabungan.count())

Found 4 items
-rw-r--r--   1 zhio supergroup          0 2026-10-10 07:45 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - zhio supergroup          0 2026-10-10 07:45 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - zhio supergroup          0 2026-10-10 07:45 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - zhio supergroup          0 2026-10-10 07:45 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


Jumlah baris terbaca dari Parquet: 4000
Jumlah baris data gabungan       : 4000


---
## F. Ringkasan Eksekutif & Rekomendasi

In [11]:
ringkasan_cabang = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_cabang.show(truncate=False)

[Stage 47:>                                                         (0 + 3) / 3]

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



### Analisis & Rekomendasi

**Kinerja cabang.** Berdasarkan tabel ringkasan, **Kopi Nusantara Semarang** adalah cabang terbaik dengan total penjualan sekitar Rp90,5 juta dari 1.393 transaksi, unggul tipis dari Magelang (Rp87,1 juta) dan Yogyakarta (Rp84,6 juta). Menariknya, rata-rata nilai transaksi tertinggi justru ada di Magelang (sekitar Rp66.900), sedangkan Semarang unggul karena volume transaksinya paling banyak. Cabang **Yogyakarta** adalah yang terlemah, terutama karena nilai per transaksinya paling rendah (sekitar Rp64.900) dan jumlah transaksinya lebih sedikit dari Semarang.

**Rekomendasi menu.** Dari bagian C, dua menu terlaris di Yogyakarta adalah **Menu-12** (Rp9,28 juta) dan **Menu-15** (Rp8,16 juta). Sebaiknya promosi di Yogyakarta difokuskan pada keduanya. Menu-12 paling layak didorong karena juga masuk tiga besar di Semarang dan Magelang, jadi permintaannya terbukti konsisten di semua cabang. Menu-15 berkategori Non-Kopi sehingga bisa menarik pelanggan yang tidak membeli kopi dan membantu menaikkan nilai per transaksi lewat paket bundling. Paket bundling Menu-12 dan Menu-15 juga bisa menaikkan jumlah item per transaksi.

**Catatan.** Selisih antar cabang cukup kecil (sekitar 7% antara Semarang dan Yogyakarta), jadi kesimpulan ini sebaiknya diuji lagi dengan data periode yang lebih panjang sebelum keputusan besar diambil. Pada data ini, Nadia ternyata menjadi kepala barista di dua cabang (Magelang dan Semarang), sehingga hasil bagian D hanya menampilkan dua barista dan total Nadia menggabungkan dua cabang sekaligus.

In [12]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
